# 0.40 — Filling the theme schema with a model that cannot see the future

Every LLM call in this project goes to a model trained in 2026. Asked to judge a theme as of
September 2022, it does not reason from 2022 evidence — it remembers how the story ended.
[DatedGPT](https://www.datedgpt.com/) ([arXiv 2603.11838](https://arxiv.org/abs/2603.11838))
removes that by construction: twelve 1.3 B Llama-architecture models, one per year from 2013
to 2024, each pretrained from scratch with a hard data cutoff, in a `base` and an
instruction-tuned variant. The 2022 vintage has never seen a token written after 2022.

**What this notebook does.** Takes the six fields of `docs/theme-schema-short.md` and asks
the model for **one field at a time, with one small prompt per field** — nothing chained,
nothing clever. Every prompt is printed in full before its answer, so there is nothing to
take on trust.

```
Entity · Event / Action · Object · Technology / Concept · Counterparty · Relationship
```

Entity resolution — which ticker "Nvidia" is — is **not** done here. That is a later step.

**The answer, measured on 15 hand-labelled headlines** (§4 reproduces it):

| field | accuracy | |
|---|---:|---|
| Entity | **73 %** | usable |
| Technology / Concept | 53 % | usable with review |
| Event / Action | 40 % | no |
| Counterparty | 40 % | no — and it fails dangerously, see §5 |
| Object | 33 % | no |
| Relationship | 7 % | no |

It names **what the headline is about** and fails at **what happened in it**.

**One thing had to be got right, and it is worth knowing why.** Asked conversationally
("Headline: … Which company is this about?"), the model ignores the question and paraphrases
the headline, for every field, however many examples it is shown. Written as a plain
completion, where three worked examples end with the field label and the model simply
continues the pattern, it answers in two or three words. Same model, same headline; the
difference is whether generation starts at the beginning of a free turn or inside an answer
that has already begun.

In [24]:
import gc
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO = "datedgpt/datedgpt-{year}-{kind}"

if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda", torch.bfloat16
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.float16      # bf16 matmul is still patchy on Metal
else:
    DEVICE, DTYPE = "cpu", torch.float32

TOK = AutoTokenizer.from_pretrained(REPO.format(year=2024, kind="instruct"))
_LOADED = {}

def model(year, kind):
    """One vintage on DEVICE — 2.7 GB downloaded on first use, then cached on disk."""
    if (year, kind) not in _LOADED:
        m = AutoModelForCausalLM.from_pretrained(REPO.format(year=year, kind=kind), dtype=DTYPE)
        _LOADED[(year, kind)] = m.to(DEVICE).eval()
    return _LOADED[(year, kind)]

def release():
    _LOADED.clear(); gc.collect()
    if DEVICE != "cpu":
        getattr(torch, DEVICE).empty_cache()

VINTAGE, KIND = 2022, "instruct"      # `base` also works and is a one-word change


@torch.no_grad()
def chat(question, year=VINTAGE, max_new_tokens=140, kind="instruct"):
    """Ask the model anything, as a conversation."""
    # The chat template already emits `<s>` as literal text, so the tokenizer must not add a
    # second one. `use_cache=True` is explicit because config.json disables the KV cache.
    text = TOK.apply_chat_template([{"role": "user", "content": question}], tokenize=False)
    enc  = TOK(text, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    out  = model(year, kind).generate(
        **enc, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False,
        repetition_penalty=1.1, eos_token_id=TOK.eos_token_id, pad_token_id=TOK.eos_token_id)
    return TOK.decode(out[0, enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()


@torch.no_grad()
def raw(prompt, year=VINTAGE, max_new_tokens=60, kind=None):
    """Send text verbatim — no template, no examples. The model just continues it."""
    enc = TOK(prompt, return_tensors="pt").to(DEVICE)
    out = model(year, kind or KIND).generate(
        **enc, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False,
        eos_token_id=TOK.eos_token_id, pad_token_id=TOK.eos_token_id)
    return TOK.decode(out[0, enc["input_ids"].shape[-1]:], skip_special_tokens=True)



@torch.no_grad()
def score(prefix, options, year=VINTAGE, kind=None):
    """Pick from a closed set: no generation, so the answer is always one of `options`.

    Returns {option: mean token log-probability}. The gap between the top two is NOT a
    usable confidence signal — measured, the most confident answers are sometimes wrong.
    """
    base, out = TOK(prefix, add_special_tokens=False)["input_ids"], {}
    for o in options:
        ids = base + TOK(" " + o, add_special_tokens=False)["input_ids"]
        lp = torch.log_softmax(model(year, kind or KIND)(
            input_ids=torch.tensor([ids]).to(DEVICE)).logits.float(), dim=-1)
        tail = ids[len(base):]
        out[o] = sum(lp[0, len(base) + j - 1, k].item()
                     for j, k in enumerate(tail)) / len(tail)
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))

print(f"device {DEVICE} · vintage {VINTAGE}-{KIND}")

device mps · vintage 2022-instruct


## Ask it anything

Three ways to talk to the model, and **which one you pick matters far more than how you word
the question**. This is measured, not a style preference.

| | how | what it is good for |
|---|---|---|
| `chat(…)` | a question, conversationally | prose *about* the headline, and world knowledge |
| `raw(…)` | text continued verbatim, after worked examples | pulling values *out of* the headline |
| `score(…)` | ranks a closed list, never generates | choosing one label from a fixed set |

**The rule: this model answers questions, it does not obey instructions.** `Summarize`,
`Explain`, `What is…`, `Who makes…` all work. `List`, `Extract`, `Return as a list`, `Answer
with one word` are all ignored — and ignored *silently*, in well-formed prose. Asked to "list
the companies in this headline" it answers **Microsoft, Alphabet, Facebook, Amazon**: it
stopped reading the text and listed the big tech firms it remembers. Asked for one word it
gives a sentence — brevity comes from examples, never from asking (§5 measures this: median
answer 8 words with no examples, 1 word with three).

So: ask `chat` about the world, use `raw` with examples for anything that must come out of the
text, and `score` when the answer belongs to a fixed list.

`year=` picks the vintage, 2013 to 2024, each blind to everything written after it. First use
of a vintage downloads 2.7 GB; `release()` frees what is resident.

**Changing `VINTAGE` changes the answers, and that is the point.** Asked which entity the
Microsoft/Bing headline is about, the 2024 vintage says `Microsoft`; the 2022 vintage says
`Bing` — it never saw the February 2023 announcement, so the link between Microsoft and
ChatGPT is not there to find. Same prompt, same code, different knowledge. §4 and §5 are
pinned to `EVAL_VINTAGE` so that turning this knob cannot quietly invalidate their numbers.

In [25]:
# ── ask anything ───────────────────────────────────────────────────────────────
h = "Microsoft to add OpenAI's ChatGPT technology to Bing search engine"

# 1 ─ PROSE: chat() honours these ──────────────────────────────────────────────
print("summary   ", chat(f"Summarize this headline in one sentence: {h}"))
meaning = chat(f"Explain what this headline means: {h}")
print("meaning   ", meaning)
h = meaning
print("sentiment ", chat(f"Headline: {h}\nIs this good news or bad news for Microsoft?"))
print("world     ", chat("Who makes the Bing search engine?"))

# 2 ─ EXTRACTION: examples, not instructions  (Entity ≈ 73 %, §4) ──────────────
shots = ("Headline: Apple Buys AI Startup to Improve Siri Voice Assistant\nEntity: Apple\n\n"
         "Headline: Pfizer Halts Trial of Obesity Pill\nEntity: Pfizer\n\n"
         "Headline: Bank Stocks Slide as Treasury Yields Climb\nEntity: Bank Stocks\n\n")
print("entity    ", raw(shots + f"Headline: {h}\nEntity:", max_new_tokens=8).split("\n")[0])

# 3 ─ CLOSED SET: score, never generate  (Predicate ≈ 60 %, vs 7 % as free text) ─
PREDICATES = ["partners with", "acquires", "invests in", "launches",
              "supplies", "sells", "sues", "none"]
pshots = ("Headline: Apple Buys AI Startup\nSubject: Apple\nObject: AI Startup\nPredicate: acquires\n\n"
          "Headline: TSMC to Supply Chips to Tesla\nSubject: TSMC\nObject: Tesla\nPredicate: supplies\n\n"
          "Headline: Pfizer Halts Trial of Obesity Pill\nSubject: Pfizer\nObject: none\nPredicate: none\n\n")
ranked = score(pshots + f"Headline: {h}\nSubject: Microsoft\nObject: OpenAI\nPredicate:", PREDICATES)
print("predicate ", next(iter(ranked)), " ", {k: round(v, 2) for k, v in list(ranked.items())[:3]})


# ✗ WHAT DOES NOT WORK — no wording fixes these, the format is wrong ────────────
# chat(f"Tell me the entities in this headline: {h}")      → echoes the headline back
# chat(f"List the companies in this headline: {h}")        → invents Alphabet, Facebook, Amazon
# chat(f"Headline: {h}\nAnswer with one word.")            → answers with a sentence
# chat(f"...the relationship (buy, sell)...")              → see §4: Relationship scores 7 %

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

summary    Microsoft will add OpenAI's chatbot technology to Bing search.
meaning    This headline means that Microsoft will add OpenAI's chatbot technology from the Bing search engine to its own search engine.
sentiment  Bad news.
world      Bing is made by Microsoft.
entity     Bing Search
predicate  launches   {'launches': -4.33, 'partners with': -4.72, 'acquires': -4.88}


## 1 · One prompt per field

Three worked examples, then the headline, then the field label. That is the whole prompt.
The examples are the same three headlines for every field — only the answers change — so
each field is taught what *kind* of answer it wants without any instruction text at all.

Two of the three examples are deliberately easy and one, *Bank Stocks Slide as Treasury
Yields Climb*, has no product, no technology and no counterparty. Its answers are `none`.
Without an example like it the model never learns that `none` is allowed, and fills every
field of every headline whether or not the headline says anything.

In [3]:
A = "Pfizer Halts Trial of Obesity Pill After Patients Report Side Effects"
B = "Apple Buys AI Startup to Improve Siri Voice Assistant"
C = "Bank Stocks Slide as Treasury Yields Climb"

EXAMPLES = {
    "Entity":               [(A, "Pfizer"), (B, "Apple"), (C, "Bank Stocks")],
    "Event / Action":       [(A, "halts trial"), (B, "buys startup"), (C, "slide")],
    "Object":               [(A, "obesity pill"), (B, "Siri voice assistant"), (C, "none")],
    "Technology / Concept": [(A, "weight-loss drug"), (B, "AI"), (C, "none")],
    "Counterparty":         [(A, "none"), (B, "AI startup"), (C, "none")],
    "Relationship":         [(A, "Pfizer stops developing its obesity pill"),
                             (B, "Apple buys a startup to improve Siri"), (C, "none")],
}
FIELDS = list(EXAMPLES)


N_SHOTS = 3        # how many worked examples go into every prompt — §5 measures the choice


def build_prompt(headline, field, n_shots=N_SHOTS):
    """One field, one prompt. The field label at the end is what primes the answer."""
    shots = [f"Headline: {h}\n{field}: {a}" for h, a in EXAMPLES[field][:n_shots]]
    return ("\n\n".join(shots) + "\n\n" if shots else "") + f"Headline: {headline}\n{field}:"


@torch.no_grad()
def ask(headline, field, n_shots=N_SHOTS, year=VINTAGE, kind=KIND, max_new_tokens=16):
    """Returns (prompt, answer). The answer is the first line the model writes."""
    prompt = build_prompt(headline, field, n_shots)
    enc = TOK(prompt, return_tensors="pt").to(DEVICE)
    gen = model(year, kind).generate(
        **enc, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False,
        eos_token_id=TOK.eos_token_id, pad_token_id=TOK.eos_token_id)
    text = TOK.decode(gen[0, enc["input_ids"].shape[-1]:], skip_special_tokens=True)
    return prompt, text.split("\n")[0].strip().strip(",.") or "none"

## Scratch — try things here

**Run the two cells above once** (they load the weights and define `ask`), then live in this
one: the model stays in the kernel, so each call is a second or two and nothing reloads.

Three things worth poking at.

*A different headline.* Change `H` and re-run — the six fields come back for it.

*A different vintage.* `ask(H, "Entity", year=2019)` asks a model that has never seen a token
written after 2019. Vintages are 2013–2024, `kind` is `"instruct"` or `"base"`; the first use
of each downloads 2.7 GB, and `release()` frees the resident ones.

*A field that does not exist yet.* Add three examples to `EXAMPLES` under any name and it is
immediately askable — three fixed examples, one of them answering `none`, is the whole recipe.

Everything below this cell is the measurement: what the prompts look like (§2, §3), how well
each field scores (§4), and whether the examples are what makes it work (§5).

In [4]:
# ── scratch ────────────────────────────────────────────────────────────────────
H = "Alphabet Unveils Willow Quantum Chip That Cuts Error Rates"

for field in FIELDS:
    print(f"{field:<22} {ask(H, field)[1]}")


# --- see the prompt behind one answer -----------------------------------------
# print(build_prompt(H, "Entity"))
# print("→", ask(H, "Entity")[1])

# --- ask an older vintage the same thing --------------------------------------
# for y in (2019, 2022, 2024):
#     print(y, "·", ask(H, "Technology / Concept", year=y)[1])
# release()

# --- invent a new field: three examples, one of them `none` -------------------
# EXAMPLES["End Market"] = [
#     ("Pfizer Halts Trial of Obesity Pill After Patients Report Side Effects", "healthcare"),
#     ("Apple Buys AI Startup to Improve Siri Voice Assistant", "consumer electronics"),
#     ("Bank Stocks Slide as Treasury Yields Climb", "none"),
# ]
# print(ask(H, "End Market")[1])

# --- send any prompt at all, bypassing the schema -----------------------------
# @torch.no_grad()
# def raw(prompt, year=VINTAGE, kind=KIND, max_new_tokens=60):
#     enc = TOK(prompt, return_tensors="pt").to(DEVICE)
#     out = model(year, kind).generate(**enc, max_new_tokens=max_new_tokens, use_cache=True,
#                                      do_sample=False, eos_token_id=TOK.eos_token_id,
#                                      pad_token_id=TOK.eos_token_id)
#     return TOK.decode(out[0, enc["input_ids"].shape[-1]:], skip_special_tokens=True)
#
# print(raw(f"Headline: {H}\nSector:"))

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Entity                 Alphabet
Event / Action         unveils quantum chip
Object                 quantum chip
Technology / Concept   quantum computing
Counterparty           Alphabet, Nvidia, and Qualcomm
Relationship           Alphabet unveils quantum chip


## 2 · One headline, six fields, one at a time

Prompt in full, then the answer. Six separate calls; the model never sees its own previous
answers.

In [5]:
HEADLINE = "Microsoft to Add OpenAI's ChatGPT Technology to Bing Search Engine"

for field in FIELDS:
    prompt, answer = ask(HEADLINE, field)
    print("═" * 92)
    print(f"FIELD:  {field}\n")
    print("PROMPT")
    print("┌" + "─" * 91)
    for line in prompt.split("\n"):
        print("│ " + line)
    print("└" + "─" * 91)
    print(f"ANSWER  →  {answer}\n")

════════════════════════════════════════════════════════════════════════════════════════════
FIELD:  Entity

PROMPT
┌───────────────────────────────────────────────────────────────────────────────────────────
│ Headline: Pfizer Halts Trial of Obesity Pill After Patients Report Side Effects
│ Entity: Pfizer
│ 
│ Headline: Apple Buys AI Startup to Improve Siri Voice Assistant
│ Entity: Apple
│ 
│ Headline: Bank Stocks Slide as Treasury Yields Climb
│ Entity: Bank Stocks
│ 
│ Headline: Microsoft to Add OpenAI's ChatGPT Technology to Bing Search Engine
│ Entity:
└───────────────────────────────────────────────────────────────────────────────────────────
ANSWER  →  Microsoft

════════════════════════════════════════════════════════════════════════════════════════════
FIELD:  Event / Action

PROMPT
┌───────────────────────────────────────────────────────────────────────────────────────────
│ Headline: Pfizer Halts Trial of Obesity Pill After Patients Report Side Effects
│ Event / Action: hal

## 3 · The same six questions on other headlines

The prompts are identical apart from the last line, so from here only the answers are shown.
The last three headlines are the hard kind the corpus is mostly made of: market mechanics
with no technology in them, and a piece of commentary with no company in it.

In [6]:
HEADLINES = [
    "Microsoft to Add OpenAI's ChatGPT Technology to Bing Search Engine",
    "IonQ Signs Deal With Hyundai to Use Quantum Computing for Battery Research",
    "Novo Nordisk Raises Outlook as Ozempic Sales Beat Estimates",
    "Alphabet Unveils Willow Quantum Chip That Cuts Error Rates",
    "Nvidia Options Imply 9.8% Share Move Post Earnings",
    "Cathie Wood Offloads Nvidia Ahead of Third-Quarter Results",
    "Generative AI Could Solve the Productivity Paradox",
]

for h in HEADLINES:
    print(f"\n{h}\n" + "─" * len(h))
    for field in FIELDS:
        print(f"   {field:<22} {ask(h, field)[1]}")


Microsoft to Add OpenAI's ChatGPT Technology to Bing Search Engine
──────────────────────────────────────────────────────────────────
   Entity                 Microsoft
   Event / Action         adds technology
   Object                 Bing search engine
   Technology / Concept   AI
   Counterparty           Microsoft
   Relationship           Microsoft to add OpenAI's ChatGPT technology to Bing search

IonQ Signs Deal With Hyundai to Use Quantum Computing for Battery Research
──────────────────────────────────────────────────────────────────────────
   Entity                 Hyundai
   Event / Action         announces deal
   Object                 hyundai
   Technology / Concept   quantum computing
   Counterparty           Hyundai
   Relationship           IonQ signs deal with Hyundai

Novo Nordisk Raises Outlook as Ozempic Sales Beat Estimates
───────────────────────────────────────────────────────────
   Entity                 Novo Nordisk
   Event / Action         announces ac

## 4 · Does it do it *well*?

Eyeballing is not an answer, so: fifteen headlines labelled by hand — the seven above and
eight more — scored field by field. Scoring is lenient on wording and strict on `none`:
case, punctuation and articles are ignored and a prediction counts if it contains the gold
answer or is contained by it, so "Booking" matches "Booking.com". But `none` matches only
`none` — a filler that never says "the headline does not say" is not extracting, it is
filling in a form.

In [7]:
import re, unicodedata

EVAL_VINTAGE = 2024      # §4 and §5 are pinned here, so changing VINTAGE above
                         # cannot silently invalidate the numbers stored below

GOLD = [
    ("Microsoft to Add OpenAI's ChatGPT Technology to Bing Search Engine",
     ["Microsoft", "add", "Bing Search Engine", "ChatGPT", "OpenAI",
      "Microsoft adds ChatGPT to Bing"]),
    ("Booking.com to Launch AI Planner Partially Powered by ChatGPT",
     ["Booking.com", "launch", "AI Planner", "ChatGPT", "none",
      "Booking.com launches an AI planner"]),
    ("Palantir Gains After Posting Rosy Forecast, Readies AI Tool",
     ["Palantir", "readies", "AI Tool", "AI", "none", "Palantir prepares an AI tool"]),
    ("Nvidia Jumps as AI Demand Boosts Results, Forecast",
     ["Nvidia", "jumps", "Results", "AI", "none", "AI demand lifts Nvidia results"]),
    ("Google’s AI Chatbot Seeks to Avoid Traps That Ensnared Microsoft",
     ["Google", "seeks to avoid", "AI Chatbot", "AI", "Microsoft",
      "Google avoids Microsoft's chatbot traps"]),
    ("Meta Shares Edge Higher as UBS Hikes PT on AI Boost",
     ["Meta", "edge higher", "Shares", "AI", "UBS", "UBS raises Meta's price target on AI"]),
    ("SoftBank Group Climbs as Optimism Builds Over Arm IPO",
     ["SoftBank", "climbs", "Arm IPO", "none", "Arm", "SoftBank rises on Arm IPO optimism"]),
    ("IonQ Signs Deal With Hyundai to Use Quantum Computing for Battery Research",
     ["IonQ", "signs deal", "Battery Research", "Quantum Computing", "Hyundai",
      "IonQ and Hyundai use quantum computing for batteries"]),
    ("Alphabet Unveils Willow Quantum Chip That Cuts Error Rates",
     ["Alphabet", "unveils", "Willow Quantum Chip", "Quantum", "none",
      "Alphabet unveils a quantum chip"]),
    ("Novo Nordisk Raises Outlook as Ozempic Sales Beat Estimates",
     ["Novo Nordisk", "raises outlook", "Ozempic Sales", "Ozempic", "none",
      "Ozempic sales lift Novo Nordisk's outlook"]),
    ("Nvidia Options Imply 9.8% Share Move Post Earnings",
     ["Nvidia", "none", "Options", "none", "none", "none"]),
    ("Cathie Wood Offloads Nvidia Ahead of Third-Quarter Results",
     ["Cathie Wood", "offloads", "Nvidia", "none", "Nvidia", "Cathie Wood sells Nvidia"]),
    ("Chipmakers Outperform as AMD Results Lift Nvidia and Intel",
     ["AMD", "results", "none", "none", "Nvidia", "AMD results lift other chipmakers"]),
    ("First Quantum Minerals Jumps Amid Barrick Takeover Interest",
     ["First Quantum Minerals", "jumps", "none", "none", "Barrick",
      "Barrick may bid for First Quantum"]),
    ("Generative AI Could Solve the Productivity Paradox",
     ["none", "none", "none", "Generative AI", "none", "none"]),
]

_ARTICLES = {"the", "a", "an", "of", "to", "in", "for", "s", "null"}

def _norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    w = [x for x in s.split() if x not in _ARTICLES]
    return " ".join(w)

def correct(pred, gold):
    """Lenient on wording, strict on `none`."""
    p, g = _norm(pred) or "none", _norm(gold) or "none"
    if g == "none" or p == "none":
        return p == g
    return p == g or g in p or p in g


rows = []
for headline, gold in GOLD:
    for field, g in zip(FIELDS, gold):
        pred = ask(headline, field, year=EVAL_VINTAGE)[1]
        rows.append(dict(headline=headline, field=field, gold=g, answer=pred,
                         ok=correct(pred, g)))
R = pd.DataFrame(rows)
release()

R.groupby("field", sort=False).ok.mean().round(2).to_frame("accuracy")

,accuracy
field,
Entity,0.73
Event / Action,0.40
Object,0.33
Technology / Concept,0.53
Counterparty,0.40
Relationship,0.07


In [8]:
print(f"overall {R.ok.mean():.0%} of {len(R)} answers\n")
for field in FIELDS:
    d = R[R.field == field]
    print(f"── {field}  ({d.ok.mean():.0%})")
    for _, r in d.iterrows():
        print(f"   {'  ' if r.ok else '✗ '}{r.headline[:48]:<50} gold={r.gold[:20]:<22} → {r.answer}")
    print()

overall 41% of 90 answers

── Entity  (73%)
     Microsoft to Add OpenAI's ChatGPT Technology to    gold=Microsoft              → Microsoft
     Booking.com to Launch AI Planner Partially Power   gold=Booking.com            → Booking.com
     Palantir Gains After Posting Rosy Forecast, Read   gold=Palantir               → Palantir
   ✗ Nvidia Jumps as AI Demand Boosts Results, Foreca   gold=Nvidia                 → AI
     Google’s AI Chatbot Seeks to Avoid Traps That En   gold=Google                 → Google
     Meta Shares Edge Higher as UBS Hikes PT on AI Bo   gold=Meta                   → Meta
     SoftBank Group Climbs as Optimism Builds Over Ar   gold=SoftBank               → SoftBank Group
   ✗ IonQ Signs Deal With Hyundai to Use Quantum Comp   gold=IonQ                   → Hyundai
     Alphabet Unveils Willow Quantum Chip That Cuts E   gold=Alphabet               → Alphabet
     Novo Nordisk Raises Outlook as Ozempic Sales Bea   gold=Novo Nordisk           → Novo Nordisk
     

## 5 · Are the examples doing the work?

The three worked examples sit inside every prompt, which is a fair thing to be suspicious of.
Two things to say about it, one by argument and one by measurement.

**By argument:** the examples are *fixed*. The same three headlines appear in every prompt for
a given field, and none of them is ever the headline being asked about. They cannot leak the
answer, because they do not contain it. What they can leak is the *shape* of the answer.

**By measurement:** run the whole gold set again at 0, 1, 2 and 3 examples and see what
changes.

In [9]:
sweep = []
for n in (0, 1, 2, 3):
    for headline, gold in GOLD:
        for field, g in zip(FIELDS, gold):
            a = ask(headline, field, n_shots=n, year=EVAL_VINTAGE)[1]        # ask() returns (prompt, answer)
            sweep.append(dict(n_shots=n, field=field, answer=a, ok=correct(a, g)))
S = pd.DataFrame(sweep)
release()

by_field = S.pivot_table(index="field", columns="n_shots", values="ok", sort=False)
by_field.assign(**{"median words": S.assign(w=S.answer.str.split().str.len())
                    .groupby("field", sort=False).w.median()}).round(2)

n_shots,0,1,2,3,median words
field,,,,,
Entity,0.73,0.67,0.80,0.73,2.0
Event / Action,0.53,0.47,0.27,0.40,3.0
Object,0.13,0.40,0.40,0.33,3.0
Technology / Concept,0.33,0.27,0.47,0.53,2.0
Counterparty,0.33,0.47,0.27,0.40,1.0
Relationship,0.00,0.00,0.07,0.07,7.0


In [10]:
print("accuracy overall     ", {n: round(v, 2) for n, v in S.groupby("n_shots").ok.mean().items()})
words = S.answer.str.split().str.len()
print("median answer length ",
      S.assign(w=words).groupby("n_shots").w.median().to_dict(), "words\n")
print("what zero-shot answers look like (Entity):")
for r in [r for r in sweep if r["n_shots"] == 0 and r["field"] == "Entity"][:5]:
    print(f"   {r['answer']}")

accuracy overall      {0: 0.34, 1: 0.38, 2: 0.38, 3: 0.41}
median answer length  {0: 8.0, 1: 2.5, 2: 2.0, 3: 1.0} words

what zero-shot answers look like (Entity):
   Microsoft and OpenAI
   Booking.com to Launch AI Planner Partially Powered
   Palantir Technologies Inc
   Nvidia Corp
   The New York Times


**The examples buy format, not knowledge.** Accuracy moves from 34 % to 41 % across the whole
range — a real gain, but small, and per-field the differences are mostly noise: at fifteen
headlines a single answer is worth 6.7 points, and `Event / Action` is actually *best* with no
examples at all. `Entity` scores 73 % with zero examples, exactly what it scores with three.
`Headline: … / Entity:` is apparently enough on its own.

What collapses is the length: the median answer goes from **8 words with no examples to 1 word
with three**. That is the whole job the examples are doing. Without them the model answers
"Booking.com to Launch AI Planner Partially Powered" — the headline back again — or invents
"The New York Times" for a headline about Google. Those are not near-misses of a right answer,
they are answers to a different question, and the examples are what tell the model which
question was asked.

So: keep the examples, but do not credit them with the result. If a field scores badly here it
is not because it was shown too few examples — `Relationship` is at 0–7 % throughout. And when
this is extended to new fields, three fixed examples that include one `none` are the whole
recipe.

## 6 · What the numbers say

Read the per-field accuracies, not the overall one: the fields are different tasks that
happen to share a prompt shape, and averaging them hides the only thing worth knowing —
**which fields are usable now and which are not.**

The shape of the result is that the model is good at naming *what the headline is about* and
poor at describing *what happened in it*. `Entity` and `Technology / Concept` are lookups:
the answer is a noun sitting in the text, and a 1.3 B model finds it. `Event / Action`,
`Object` and `Relationship` require deciding what the sentence asserts, which is where a
model this size stops being reliable — its errors there are not near-misses but restatements
of the whole headline.

`Counterparty` is the field to watch, because it fails in the most dangerous way: when the
headline has no second company, the model tends to repeat the entity rather than answer
`none`. A wrong `none` is a missing edge; a repeated entity is a **false** edge, and
downstream nothing distinguishes it from a true one. If this field is used, it needs a check
that the answer is not simply the entity again.

**Two caveats on the numbers themselves.** Fifteen headlines is a small gold set: treat the
extremes as real and ignore small differences between fields. And the schema was written for
*articles* — `docs/theme-schema-long.md` assumes a full body with derived fields and
confidence scores — so feeding a first paragraph instead of a headline alone would give the
model the context that `Counterparty` and `Object` are currently guessing at. That is the
cheapest next experiment, and it needs no new model.

---

## Appendix · Why a dated model at all

One question, five vintages, to show what the cutoff actually does.

In [11]:
import textwrap

for y in [2013, 2019, 2022, 2023, 2024]:
    print(f"\n{'─' * 4} {y} {'─' * 66}")
    print(textwrap.fill(chat("What is ChatGPT, and which company makes it?", year=y), 78)
          or "(empty)")
    release()


──── 2013 ──────────────────────────────────────────────────────────────────
ChatGLT is a new open-source version of the GHC compiler. It is made by
NexGen.

──── 2019 ──────────────────────────────────────────────────────────────────
ChatGPT is a free, open-source chat client for the Android operating system.
It was developed by the same team that created the popular instant messaging
app Viber.

──── 2022 ──────────────────────────────────────────────────────────────────
ChatGPT is a free, open-source chatbot platform that allows businesses to
create chatbots for customer support. It was developed by the team behind the
popular Slack app.

──── 2023 ──────────────────────────────────────────────────────────────────
ChatGPT is an artificial intelligence chatbot developed by OpenAI. It can
answer questions in a conversational way from users, providing information on
various topics. The company that makes it is OpenAI.

──── 2024 ────────────────────────────────────────────────────────

2023 and 2024 answer correctly. 2013 does not recognise the string and repairs it into
something it knows — "ChatGLT … the GHC compiler". The ones that matter are 2019 and 2022,
which answer **fluently, in the right register, and completely wrongly**: an open-source chat
client for Android "developed by the same team that created Viber", a chatbot platform
"developed by the team behind the popular Slack app".

Out of horizon these models do not hedge, they invent, and nothing in the tone marks the
difference. Which is the argument for §4: whether a field works is something to measure, not
something to ask the model about.